# 🔧 Run a Task DAG — the plan executor in embryo

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 6 §6.4 · type: walkthrough*

**The promise:** by the end you can take an agent plan expressed as a dependency dict and run it concurrently — every ready batch at once — in latency equal to the graph's *depth*, not its size. This is the capstone's scheduling core, built from scratch.

Runs fully offline and free: the "work" each task does is a mocked `asyncio.sleep`, so there are no model calls and the timings are deterministic.

## 🧠 Why this matters

An agent's plan is a **DAG**: research before drafting, fact-checking before review, both before the final write. The question "what can run, and what can run *in parallel*?" is exactly **topological sorting** — and the standard library ships it.

Get this wrong and you pay the graph's *size* in latency: a planner that serializes 12 independent-where-possible subtasks runs in 12 model-call latencies. Get it right and you pay the graph's *depth* — often 3 or 4 — because each ready batch runs concurrently with `asyncio.gather`. Correctness comes from the topological order; speed comes from `gather`. That single trade is most of what makes multi-agent orchestration (Part V) economically viable. See §6.4.

## Objectives & prereqs

**By the end you can:**
- Model an agent plan as `deps: dict[str, set[str]]` (task → the tasks it depends on).
- Build `run_plan` with `graphlib.TopologicalSorter`: `prepare → while is_active → get_ready → gather → done`.
- Explain why wall-clock time tracks the graph's **depth**, not its task count.
- Add a bounded `asyncio.Semaphore` so a wide ready-batch doesn't stampede your rate limit.
- Detect a dependency **cycle** — `prepare()` raises — and say why DAGs must be acyclic.

**Prereqs:** `06-01` (cost curves, the daily structures); Chapter 4 async — `asyncio.gather` and the bounded `Semaphore`.

**Packages:** the standard library only (`asyncio`, `graphlib`, `time`). Nothing to install beyond `requirements.txt`.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import os
import random
import time
from graphlib import CycleError, TopologicalSorter

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): each task is a mocked asyncio.sleep instead of a real model
# call, so the whole notebook runs offline, free, and deterministically. There is
# no live path here -- the lesson is the SCHEDULER, and a sleep stands in perfectly
# for "an awaitable unit of work that takes ~one model-call latency."
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(6)  # determinism for anything stochastic

# One "model-call latency," simulated. Small so CI is fast; the RATIOS are the point.
TASK_LATENCY = 0.10  # seconds per task (a stand-in for a real call)

print(f"MOCK mode: {MOCK}  (offline, free; tasks are mocked asyncio.sleep)")
print(f"simulated per-task latency = {TASK_LATENCY:.2f}s")

## 1 · Three classics, in production clothes

Before the build, place it on the map. The algorithmic problems you actually meet in AI systems are the textbook ones wearing a trench coat:

- **Search** = nearest-neighbor. Vector retrieval over n embeddings is exact O(n) per query (fine to ~a million vectors) or, beyond that, approximate (ANN/HNSW) — sub-linear, by greedily walking a layered graph from coarse to fine. The structure decides the cost.
- **Ranking** = heaps + merge. "Keep the best k as scores stream in," fusing several retrievers — recognize "best k of many" and you never sort a million items to take ten (you saw `heapq.nlargest` in `06-01`).
- **Scheduling** = topological sort. "What can run, and what can run in parallel?" — this notebook's build.

We focus on the third, because it is the capstone's plan executor.

## 2 · Model a plan as a dependency dict

The humble, standard representation of a graph is a dict mapping each node to its neighbors. For a plan, that means each task → the set of tasks that must finish **before** it. Here is a 12-task content-pipeline plan; read it as "this needs those."

In [ ]:
# deps[task] = set of tasks that must complete BEFORE `task` can run.
# A small 12-node plan: three independent research legs fan out, get synthesized,
# drafted in parallel sections, fact-checked, reviewed, and published.
deps: dict[str, set[str]] = {
    "research_market":   set(),
    "research_tech":     set(),
    "research_legal":    set(),
    "synthesize":        {"research_market", "research_tech", "research_legal"},
    "outline":           {"synthesize"},
    "draft_intro":       {"outline"},
    "draft_body":        {"outline"},
    "draft_conclusion":  {"outline"},
    "fact_check":        {"draft_intro", "draft_body", "draft_conclusion"},
    "copy_edit":         {"draft_intro", "draft_body", "draft_conclusion"},
    "review":            {"fact_check", "copy_edit"},
    "publish":           {"review"},
}

print(f"{len(deps)} tasks. A few dependencies:")
for task in ["synthesize", "fact_check", "publish"]:
    print(f"  {task:18s} needs -> {sorted(deps[task]) or '(nothing; a root)'}")

## 3 · 🔧 Build `run_plan` with `TopologicalSorter`

This is the chapter's build, verbatim in spirit. The five-line dance is the whole idea: **prepare** the sorter (this also rejects cycles), then loop while it `is_active` — pull every task whose dependencies are satisfied with `get_ready`, run that batch concurrently with `gather`, and mark each one `done` so the next batch unlocks.

In [ ]:
async def run_plan(deps: dict[str, set[str]], run_task) -> None:
    """Execute a task DAG, running every ready batch concurrently.

    deps maps task -> set of tasks it depends on, e.g.
    {"draft": {"research"}, "review": {"draft", "fact_check"}}
    """
    ts = TopologicalSorter(deps)
    ts.prepare()                      # also raises on dependency cycles
    while ts.is_active():
        batch = ts.get_ready()        # all tasks whose deps are satisfied
        await asyncio.gather(*(run_task(t) for t in batch))
        for t in batch:
            ts.done(t)

Now the work itself. In production each task is a model call; here a `run_task` that sleeps for one latency and records *when* it ran. Recording the batch (the "depth level") lets us *see* the scheduler group independent tasks together.

In [ ]:
# A mock task: sleep one latency, record the order and the batch it landed in.
# This is the only place a real system would call the model; the scheduler above
# does not change at all when you swap this stub for an API call.
run_log: list[tuple[float, str]] = []


def make_run_task(log: list):
    t0 = time.perf_counter()

    async def run_task(name: str) -> None:
        await asyncio.sleep(TASK_LATENCY)        # stands in for a model call
        log.append((time.perf_counter() - t0, name))

    return run_task


# A sanity check: the same dict, run by a SEQUENTIAL executor, for the comparison.
async def run_plan_sequential(deps, run_task) -> None:
    """Correct but naive: one task at a time, in a valid topological order."""
    ts = TopologicalSorter(deps)
    for task in ts.static_order():               # a flat, dependency-safe order
        await run_task(task)


print("run_plan (batched) and run_plan_sequential (naive) are both ready.")

## 4 · 🔮 Predict: 12 tasks — how long?

Each task takes `TASK_LATENCY` (0.10s). The **sequential** executor runs all 12 one after another. The **batched** executor runs every ready batch concurrently.

**Predict before running:**
1. The sequential wall-clock — that one's arithmetic.
2. The batched wall-clock. *Hint:* count the **depth** of the plan — the longest chain `research_* → synthesize → outline → draft_* → fact_check/copy_edit → review → publish`. How many sequential *levels* is that? The batched time should be roughly `depth × latency`, regardless of the 12 task count.

Write both guesses down, then run.

In [ ]:
# --- Sequential: pays the graph's SIZE (12 latencies) ---
run_log.clear()
start = time.perf_counter()
await run_plan_sequential(deps, make_run_task(run_log))
t_seq = time.perf_counter() - start

# --- Batched by depth: pays the graph's DEPTH ---
batched_log: list = []
start = time.perf_counter()
await run_plan(deps, make_run_task(batched_log))
t_batched = time.perf_counter() - start

n = len(deps)
print(f"sequential : {t_seq:5.2f}s  (~{n} latencies = the graph's SIZE)")
print(f"batched    : {t_batched:5.2f}s  (~depth latencies = the graph's DEPTH)")
print(f"speedup    : ~{t_seq / t_batched:.1f}x  for the SAME 12 tasks, SAME work")

In [ ]:
# Make the "depth" concrete: group the batched run into the concurrent waves the
# scheduler ran. Tasks that finished within the same latency window ran together.
# We cluster by finish time (robust to scheduler jitter), then number waves 1..depth.
waves: list[list[str]] = []
prev_t = None
for finish_t, name in sorted(batched_log):
    # A gap of more than half a latency means a new dependency wave began.
    if prev_t is None or (finish_t - prev_t) > TASK_LATENCY / 2:
        waves.append([])
    waves[-1].append(name)
    prev_t = finish_t

print(f"the plan ran in {len(waves)} concurrent waves (its depth):\n")
for i, names in enumerate(waves, start=1):
    print(f"  wave {i}: {len(names)} task(s) in parallel -> {sorted(names)}")

**What you just saw.** The three `research_*` tasks ran together in wave 1; the three `draft_*` together; `fact_check` and `copy_edit` together. Twelve tasks collapsed into a handful of concurrent waves, so wall-clock time tracks the plan's **depth**, not its task count. The sequential executor did the identical work but paid for all twelve latencies in series. Correctness came from the topological order; the speedup came entirely from `gather`.

## 5 · Bound the concurrency with a `Semaphore`

A wide ready-batch is a hazard. If 50 tasks become ready at once, `gather` fires all 50 model calls simultaneously — straight into your provider's rate limit (429s) or your wallet. The Chapter 4 fix is a bounded `asyncio.Semaphore`: at most `limit` tasks in flight at any moment, the rest waiting their turn. Same scheduler, one wrapper.

In [ ]:
async def run_plan_bounded(deps, run_task, limit: int = 4) -> None:
    """Like run_plan, but never more than `limit` tasks in flight at once."""
    sem = asyncio.Semaphore(limit)

    async def guarded(task):
        async with sem:               # acquire a slot; block past `limit` in flight
            await run_task(task)

    ts = TopologicalSorter(deps)
    ts.prepare()
    while ts.is_active():
        batch = ts.get_ready()
        await asyncio.gather(*(guarded(t) for t in batch))
        for t in batch:
            ts.done(t)


# A plan with a DELIBERATELY wide batch: 8 independent leaves -> one fan-in.
wide = {f"leaf_{i}": set() for i in range(8)}
wide["merge"] = {f"leaf_{i}" for i in range(8)}

for limit in (2, 8):
    start = time.perf_counter()
    await run_plan_bounded(wide, make_run_task([]), limit=limit)
    elapsed = time.perf_counter() - start
    # 8 leaves at `limit`-at-a-time = ceil(8/limit) waves, then +1 for merge.
    print(f"limit={limit}: {elapsed:.2f}s  "
          f"(8 leaves in ceil(8/{limit})={-(-8 // limit)} wave(s) + merge)")

**What you just saw.** With `limit=8` the eight leaves run in one wave (~one latency); with `limit=2` they run four-at-a-time-worth in `ceil(8/2)=4` waves. The semaphore trades a little latency for *survival* under a rate limit — and in production that trade is mandatory, because 8 became 80 the moment a real plan arrived.

## 6 · ⚠️ Pitfall: a dependency cycle

A DAG must be **acyclic** — the *A* in DAG. If task A waits on B and B waits on A, no task is ever "ready": the plan can never start. `TopologicalSorter.prepare()` catches this *up front* and raises `CycleError`, naming the cycle, instead of deadlocking silently at runtime. Fail loud, fail early.

In [ ]:
# A planner bug: 'review' depends on 'draft', but 'draft' was given a dependency
# back on 'review'. Nothing can ever run. prepare() refuses to start.
cyclic = {
    "research": set(),
    "draft":    {"research", "review"},   # <-- depends on review...
    "review":   {"draft"},                # <-- ...which depends on draft. cycle!
}

try:
    await run_plan(cyclic, make_run_task([]))
except CycleError as exc:
    # graphlib raises at prepare() -- BEFORE any task runs -- and names the loop.
    print("CycleError caught at prepare() (nothing ran):")
    print("  ", exc.args[0])
    print("   cycle:", " -> ".join(exc.args[1]))

print(
    "\nWhy it matters: a silent deadlock costs you a hung worker and a 3am page.\n"
    "prepare() converts that into a loud, immediate, debuggable error -- validate\n"
    "the plan BEFORE you spend a single model call on it."
)

## 🎯 Senior lens: depth is the budget, topo order is the contract

A planner that serializes independent subtasks pays the graph's **size** in latency; one that batches by readiness pays its **depth**. On a 12-task plan that is a 3–4× user-facing speedup for free — and the work, the cost, and the correctness are all identical. That is the senior instinct this chapter is really teaching: **correctness comes from the topological order, speed from `gather`** — they are separable, and you get both.

Two judgments keep it production-safe. First, **bound the fan-out**: an unbounded `gather` over a wide batch is a self-inflicted rate-limit incident, so the semaphore is not optional once tasks make real calls. Second, **validate the plan before spending on it**: `prepare()` is cheap and a model call is not, so reject cycles (and, in practice, also impossible deps and runaway widths) at plan time, never at run time.

## 📋 Complexity & structures checklist

A self-audit you can run against any plan-executing code (mirrors the chapter's checklist):

- [ ] **Reflex check:** every loop body that scans a *growing* collection is a quadratic suspect (`in` on a list, `remove`, `+=` on strings).
- [ ] **Lookups and dedup** go through `dict`/`set`; "match two lists" means "build a dict first."
- [ ] **Queues** are `deque` in-process (Redis/SQS/Celery across processes) — never `list.pop(0)`.
- [ ] **Top-k** uses a heap (`heapq.nlargest`), not a full sort.
- [ ] **Ordered/range** queries mean trees: `bisect` in memory, B-tree indexes in the DB.
- [ ] **Workflows and plans** are graphs; cycles are checked (`graphlib`), independent branches run concurrently, and fan-out is bounded.
- [ ] **Budgets are in tokens**, counted with the model's tokenizer — not characters, not messages.
- [ ] **Model calls are the costly unit:** estimate every design in calls-per-task before optimizing anything cheaper.

## Recap

- An agent plan is a **DAG**; "what can run in parallel?" is **topological sort**, and `graphlib.TopologicalSorter` ships it in the stdlib.
- `run_plan` is five lines: `prepare → while is_active → get_ready → gather → done`. Correctness from the topo order, speed from `gather`.
- Wall-clock time tracks the plan's **depth**, not its task count — a 12-task plan can run in 3–4 concurrent waves.
- A bounded `asyncio.Semaphore` caps in-flight tasks so a wide ready-batch doesn't stampede your rate limit.
- A **cycle** makes the plan unstartable; `prepare()` raises `CycleError` up front instead of deadlocking — validate before you spend.
- This is the scheduling core Part V's multi-agent orchestration is built on.

## Exercises

Each exercise *changes* something and asks you to predict the result before you run. (Solutions arrive in `solutions/` in Phase 2.)

1. **Reshape the plan, re-predict the depth.** Make `fact_check` and `copy_edit` depend only on `draft_body` (not all three drafts). Predict how the number of concurrent waves changes, then run and group by wave to confirm.
2. **Find the critical path.** Write a function that returns the plan's *depth* (longest dependency chain) directly from `deps`, without running it. Predict it equals the wave count from section 4, then check.
3. **Tune the bound.** For the `wide` 8-leaf plan, predict the wall-clock at `limit = 1, 3, 4, 8` (think `ceil(8/limit)` waves + merge). Measure all four and plot or print the curve — where do diminishing returns start?
4. **Catch the cycle yourself.** Without `TopologicalSorter`, write a check that detects a cycle in a `deps` dict using a DFS with a "currently-on-the-stack" set. Predict what it reports for the section-6 `cyclic` plan, then confirm it agrees with `CycleError`.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

You built the toy scheduler; here's where the real one lives.

- 🎓 **Capstone:** this `run_plan` + bounded-`Semaphore` pattern is the seed of the capstone's plan executor / scheduling core. Part V's multi-agent orchestration receives agent plans as dependency dicts *exactly* like `deps` here and runs them with this same engine.
- 🔧 **Forward links:** the top-k/heap and ANN-search patterns from section 1 recur in retrieval — see the RAG pipeline ([`../../../blueprints/rag-pipeline/`](../../../blueprints/rag-pipeline/)) and Chapter 13; the DAG executor reappears in Chapter 17 (multi-agent orchestration).
- 📖 **Book:** §6.4 (search, ranking, scheduling) and the chapter's 🔧 Build. The bounded `Semaphore` comes from Chapter 4's async section — revisit it if the concurrency cap felt unfamiliar.